# 01. NumPy for images: pixels, coordinates, crops, arithmetic, and masks

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner

## What you will learn

- How pixels are addressed
- The difference between `(row, column)` and `(x, y)`
- How slicing creates a crop/ROI
- Why dtype matters during arithmetic
- How Boolean masks select pixels
- Why vectorized NumPy code is preferred over manual pixel loops


## 1. Start with a 3 × 3 image

Large images hide the indexing rules. A tiny array makes every pixel visible.

The same syntax will later work on microscopy images with millions of pixels.


In [ ]:
import numpy as np

# Create a tiny 8-bit image.
# dtype=np.uint8 intentionally matches a common image storage format.
tiny = np.array(
    [
        [0,   25,  50],
        [75, 100, 125],
        [150, 200, 255],
    ],
    dtype=np.uint8,
)

print(tiny)
print("shape:", tiny.shape)
print("dtype:", tiny.dtype)


## 2. Pixel indexing: `image[row, column]`

NumPy image indexing is:

```python
image[row, column]
```

This is one of the most important conventions to learn.

### Compare with Cartesian coordinates

In mathematics we often write `(x, y)`.  
In an image array, we usually index `(row, column)`.

- row changes vertically,
- column changes horizontally.


In [ ]:
# Center pixel: second row, second column.
# Python indexing starts at zero.
center = tiny[1, 1]
print("center:", center)

# Top-left pixel.
print("top-left:", tiny[0, 0])

# -1 means the final index along that axis.
print("bottom-right:", tiny[-1, -1])


## 3. Cropping with slicing

General pattern:

```python
crop = image[row_start:row_stop, column_start:column_stop]
```

Important: `row_stop` and `column_stop` are **excluded**.

### When is cropping useful?

- selecting a defined region of interest,
- testing an algorithm on a small region,
- excluding a known irrelevant border,
- reducing memory while developing code.

### When is cropping dangerous?

When the ROI is selected differently for every image only to make the result look better.


In [ ]:
# Rows 0 and 1 are included; row 2 is excluded.
# Columns 1 and 2 are included; column 3 is excluded.
crop = tiny[0:2, 1:3]

print(crop)
print("crop shape:", crop.shape)


## 4. Arithmetic and the `uint8` problem

`uint8` can represent only 0–255.

Before doing quantitative arithmetic, ask:

> Can this calculation produce values below 0 or above 255?

If yes, convert to a safer numerical type first.


In [ ]:
# Convert to floating point before arithmetic.
# Why? Float values are not restricted to the 0–255 integer range.
tiny_float = tiny.astype(np.float32)

# Apply one operation to every pixel.
brighter = tiny_float + 30

print(brighter)

# If we later need an 8-bit output, first restrict the values to 0..255.
safe_uint8 = np.clip(brighter, 0, 255).astype(np.uint8)

print("\nconverted safely back to uint8:")
print(safe_uint8)


## 5. Boolean masks

A Boolean mask is an array of `True` and `False`.

Example question:

> Which pixels are brighter than 100?

That becomes:

```python
mask = image > 100
```

This idea is the foundation of threshold-based segmentation.


In [ ]:
# Compare every pixel with 100.
mask = tiny > 100

print(mask)

# True behaves like "selected foreground" for many image-processing tasks.
print("selected-pixel count:", np.count_nonzero(mask))

# Boolean indexing returns only values where the mask is True.
selected_values = tiny[mask]

print("selected intensities:", selected_values)
print("mean selected intensity:", selected_values.mean())


## 6. Pixel count versus object count

This distinction is critical:

```python
np.count_nonzero(mask)
```

counts **foreground pixels**.

It does **not** count how many separate objects are present.

Object counting comes later after connected-component labeling.


## 7. Why vectorized NumPy code is preferred

A beginner may think about visiting every pixel manually with nested loops.

Instead of:

```python
for every row:
    for every column:
        check pixel
```

we usually write:

```python
mask = image > threshold
```

This is called a **vectorized operation**. It is shorter, clearer, and usually much faster.


In [ ]:
import matplotlib.pyplot as plt
import skimage as ski

# Load a larger grayscale example.
image = ski.data.moon()

# Select one rectangular region.
crop = image[150:350, 120:380]

# Display original and crop for visual confirmation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(crop, cmap="gray")
axes[1].set_title("Crop / ROI")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# Statistics help us understand the ROI before choosing later algorithms.
print("minimum :", crop.min())
print("maximum :", crop.max())
print("mean    :", crop.mean())
print("median  :", np.median(crop))
print("std     :", crop.std())


## 8. Build a mask from an image statistic

Here we use the mean only to demonstrate the syntax.

This does **not** mean the mean is always a good segmentation threshold.


In [ ]:
# Calculate one demonstration threshold.
threshold = crop.mean()

# Pixels greater than the threshold become True.
bright_mask = crop > threshold

print("threshold:", threshold)
print("fraction of pixels selected:", bright_mask.mean())

plt.figure(figsize=(6, 5))
plt.imshow(bright_mask, cmap="gray")
plt.title("Demonstration mask: crop > mean")
plt.axis("off")
plt.tight_layout()
plt.show()


## 9. Function-selection guide

| Need | Function / syntax | Why / when |
|---|---|---|
| Image dimensions | `image.shape` | Understand rows, columns, channels |
| One pixel | `image[row, col]` | Inspect a location |
| ROI/crop | `image[r0:r1, c0:c1]` | Select a rectangular region |
| Pixels meeting a rule | `image[mask]` | Boolean selection |
| Number of selected pixels | `np.count_nonzero(mask)` | Pixel count, not object count |
| Typical central value | `np.median()` | More robust to outliers than mean |
| Percentile | `np.percentile()` | Intensity landmarks |
| Restrict numerical range | `np.clip()` | Prevent invalid output range |

## Common mistakes

- using `(x, y)` mentally while indexing `(row, column)`,
- forgetting that slice stop values are excluded,
- doing unsafe arithmetic on `uint8`,
- confusing pixel count with object count,
- assuming a convenient statistic is automatically a valid segmentation threshold.

## Practice

1. Make a different crop from `ski.data.moon()`.
2. Calculate the 5th, 50th, and 95th percentiles.
3. Make a mask above the 75th percentile.
4. Display the mask.
5. Explain why it still needs validation.

## Takeaway

**Indexing tells you where. A Boolean mask tells you which pixels. NumPy statistics help you understand the data before selecting an image-processing algorithm.**
